# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ravindrathalari06/flyrank-internship-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Two paper findings + my methodology questions

### Answer
### Finding 1 — Learned model vs. baseline

The workflow reports that the learned model performed better than the hand-written baseline on Precision@50. The label comes from the observed `trend_direction` field, where `trend_direction == "down"` is treated as declining.

My methodology question is whether the validation design is strong enough to support this improvement. The client-holdout split is useful because pages from the same client are kept out of both training and testing, but the result should be treated as measured performance on this dataset rather than proof of future performance.

### Finding 2 — Client-holdout validation

The workflow uses a client-holdout split so that clients in the training set do not appear in the test set. This is a stronger test of generalization across clients than randomly splitting individual rows.

My methodology question is whether client-holdout validation is sufficient for a time-dependent claim. A time-aware validation would provide an additional check because the dataset contains historical performance windows.


In [22]:
# Section 1: Check the label source used in the paper findings

print("Label source: trend_direction")
print('Declining condition: trend_direction == "down"')

required_fields = [
    "trend_direction",
    "trend_pct",
    "client_id"
]

print("\nRequired fields present:")
for field in required_fields:
    print(f"{field}: {field in data.columns}")

print("\nObserved trend_direction values:")
print(data["trend_direction"].value_counts(dropna=False))

Label source: trend_direction
Declining condition: trend_direction == "down"

Required fields present:
trend_direction: True
trend_pct: True
client_id: True

Observed trend_direction values:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [23]:
# ML-09 Section 2
# Re-run the Week-5 model using a grouped client split

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

# Create the label exactly as in Week-5
data["is_declining_label"] = (
    data["trend_direction"] == "down"
).astype(int)

# --------------------------------------------------
# 3. EXACT Week-5 feature list
# --------------------------------------------------

feature_fields = [
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "content_type",
    "main_intent",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier",
    "age_tier_order",
    "days_since_last_update",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "impression_tier",
    "position_tier"
]

# --------------------------------------------------
# 4. Grouped client split
# --------------------------------------------------

clients = data["client_id"].unique()

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=123
)

train_df = data[data["client_id"].isin(train_clients)].copy()
test_df = data[data["client_id"].isin(test_clients)].copy()

print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

overlap = set(train_clients) & set(test_clients)
print("Client overlap:", len(overlap))

# --------------------------------------------------
# 5. Prepare X and y
# --------------------------------------------------

X_train = train_df[feature_fields]
X_test = test_df[feature_fields]

y_train = train_df["is_declining_label"]
y_test = test_df["is_declining_label"]

numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = [
    c for c in feature_fields
    if c not in numeric_features
]

# --------------------------------------------------
# 6. Same Week-5 preprocessing
# --------------------------------------------------

preprocessor = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        numeric_features
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    )
])

# --------------------------------------------------
# 7. Same Week-5 Logistic Regression
# --------------------------------------------------

model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

model.fit(X_train, y_train)

# --------------------------------------------------
# 8. Predict
# --------------------------------------------------

scores = model.predict_proba(X_test)[:, 1]

# --------------------------------------------------
# 9. Precision@50
# --------------------------------------------------

def precision_at_50(y_true, scores):
    top_50 = scores.argsort()[::-1][:50]
    return y_true.iloc[top_50].mean()

honest_precision = precision_at_50(
    y_test.reset_index(drop=True),
    scores
)

# --------------------------------------------------
# 10. Show BEFORE and AFTER
# --------------------------------------------------

print("\n===== BEFORE / AFTER ====")
print("Week-5 original Precision@50: 1.00")
print(
    "ML-09 honest grouped Precision@50:",
    round(honest_precision, 4)
)
print("Test clients:", len(test_clients))
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

overlap = set(train_clients).intersection(set(test_clients))
print("Client overlap:", len(overlap))

assert len(overlap) == 0

print("\nResult: No client appears in both train and test.")

Training clients: 25
Test clients: 7
Training rows: 26496
Test rows: 3504
Client overlap: 0

===== BEFORE / AFTER ====
Week-5 original Precision@50: 1.00
ML-09 honest grouped Precision@50: 1.0
Test clients: 7
Training rows: 26496
Test rows: 3504
Client overlap: 0

Result: No client appears in both train and test.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-09 Section 3: Leakage audit

# Target-related fields that should NOT be used as model features
target_related_fields = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

# Identity/group fields that should not be model features
identity_fields = [
    "content_id",
    "client_id"
]

# Check whether any forbidden fields are present
# in the final Week-5 feature set.

leakage_fields = [
    col for col in feature_fields
    if col in target_related_fields + identity_fields
]

print("Final feature count:", len(feature_fields))
print("Potential leakage fields found:", leakage_fields)

if len(leakage_fields) == 0:
    print("\nPASS: No direct target/identity leakage fields found.")
else:
    print("\nWARNING: Potential leakage fields detected.")

Final feature count: 38
Potential leakage fields found: []

PASS: No direct target/identity leakage fields found.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

ANSWER
### Original Bold Claim

> The learned model outperforms the hand-written baseline for identifying declining content.

### Safer Claim Rewrite

> **Observed:** In this dataset, the learned Logistic Regression model achieved a measured Precision@50 of **1.00** under the evaluated grouped client-holdout split, compared with the Week-4 hand-written baseline of **0.46**. The result is directional evidence that the learned model may provide useful decision support for prioritizing potentially declining content. However, this result is measured on the evaluated dataset and validation splits and should not be interpreted as proof of future or universal performance.


In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: Code-backed check for the rewritten claim

before_precision = 1.00
after_precision = 1.00
baseline_precision = 0.46

print("Week-5 learned model Precision@50:", before_precision)
print("ML-09 honest grouped Precision@50:", after_precision)
print("Week-4 baseline Precision@50:", baseline_precision)

print("\nObserved change from Week-5 to ML-09:",
      after_precision - before_precision)

print("\nInterpretation:")
print("The measured Precision@50 remained unchanged under the tested grouped split.")
print("This supports a directional decision-support claim, not a guarantee of future performance.")

Week-5 learned model Precision@50: 1.0
ML-09 honest grouped Precision@50: 1.0
Week-4 baseline Precision@50: 0.46

Observed change from Week-5 to ML-09: 0.0

Interpretation:
The measured Precision@50 remained unchanged under the tested grouped split.
This supports a directional decision-support claim, not a guarantee of future performance.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.